# 01 · Data: explore, clean, split

In [ ]:
import os, sys
REPO_URL = "https://github.com/YUPWE/car-models.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/car-models"
    if not os.path.exists(ROOT):
        !git clone {REPO_URL} {ROOT}
    os.chdir(ROOT)
    !git pull -q
    !pip install -q imagehash==4.3.2
    os.environ["CARS_DATA_DIR"] = "/content/cars_data"
else:
    ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(ROOT)
sys.path.insert(0, ROOT)
os.makedirs("results", exist_ok=True)
os.makedirs("weights", exist_ok=True)
QUICK = os.environ.get("QUICK", "0") == "1"
print("Working in", os.getcwd(), "| QUICK =", QUICK)

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from src import data

data.download()
df = data.load_meta()
names = data.model_names(df)
json.dump(names, open("results/model_names.json", "w"), indent=0)
print(len(df), "images,", len(names), "car models,", df.make.nunique(), "makes, years", df.year.min(), "-", df.year.max())

per_model = df.model.value_counts()
print("images per model: min", per_model.min(), "| median", int(per_model.median()), "| max", per_model.max())
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
per_model.plot.hist(bins=30, ax=ax[0], title="Images per car model")
df.make.value_counts().plot.bar(ax=ax[1], title="Images per make", fontsize=6)
plt.tight_layout(); plt.savefig("results/class_counts.png", dpi=120); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(18, 6)); axes = axes.ravel()
for a, (_, row) in zip(axes, df.sample(12, random_state=3).iterrows()):
    a.imshow(Image.open(f"{data.IMG_DIR}/{row.path}")); a.axis("off"); a.set_title(row.model[:32], fontsize=8)
plt.tight_layout(); plt.savefig("results/examples.png", dpi=100); plt.show()

## Cleaning

In [ ]:
short = df[["orig_w", "orig_h"]].min(axis=1)
tiny = set(df.index[short < 64])
print(len(tiny), "images with shorter side < 64 px")

hashes = data.perceptual_hashes(df.path)
pairs = data.near_duplicate_pairs(hashes, max_dist=4)
conflicts = [(i, j) for i, j in pairs if df.model[i] != df.model[j]]
print(len(pairs), "near-duplicate pairs;", len(conflicts), "of them labelled as two different models")

show = conflicts[:6]
fig, axes = plt.subplots(len(show), 2, figsize=(8, 2.8 * len(show)), squeeze=False)
for r, (i, j) in enumerate(show):
    for k, idx in enumerate([i, j]):
        axes[r, k].imshow(Image.open(f"{data.IMG_DIR}/{df.path[idx]}")); axes[r, k].axis("off")
        axes[r, k].set_title(df.model[idx], fontsize=9)
plt.tight_layout(); plt.savefig("results/label_conflicts.png", dpi=100); plt.show()

In [ ]:
drop = sorted(tiny | {k for p in conflicts for k in p})
keep = [i for i in df.index if i not in set(drop)]
new_id = {old: new for new, old in enumerate(keep)}
clean = df.loc[keep].reset_index(drop=True)
clean_pairs = [(new_id[i], new_id[j]) for i, j in pairs if i in new_id and j in new_id]
groups = data.duplicate_groups(len(clean), clean_pairs)
print(f"Dropped {len(drop)} images -> {len(clean)} remain; {len(clean_pairs)} duplicate pairs will be kept inside one split")

## Split: every model in train, val and test; duplicates never cross splits

In [ ]:
split_df = data.make_splits(clean, groups, seed=42)
print(split_df.split.value_counts())
print(split_df.split.value_counts(normalize=True).round(3))
counts = split_df.groupby(["model_id", "split"]).size().unstack(fill_value=0)
print("models missing from a split:", int((counts == 0).any(axis=1).sum()))
print("test images per model: min", counts.test.min(), "| median", int(counts.test.median()), "| max", counts.test.max())
assert split_df.groupby("dup_group").split.nunique().max() == 1, "a duplicate group is in two splits!"
print("OK: no duplicate photo appears in two splits")
data.save_splits(split_df)
print(os.listdir("splits"))